## Install Required Libraries

In [14]:
!pip install nemo-curator dask distributed fasttext datasets


Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com

[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip


## Load UGPhysics Dataset from Hugging Face

In [15]:
from datasets import load_dataset

# Choose ONE physics domain (config)
hf_dataset = load_dataset(
    "UGPhysics/ugphysics",
    name="AtomicPhysics",   
    split="en"
)

print(hf_dataset)
hf_dataset[0]


Dataset({
    features: ['index', 'domain', 'subject', 'topic', 'problem', 'solution', 'answers', 'answer_type', 'unit', 'is_multiple_answer', 'language', 'level'],
    num_rows: 915
})


{'index': 575,
 'domain': 'Modern Physics',
 'subject': 'Atomic Physics',
 'topic': 'Atomic and Molecular Physics',
 'problem': "A certain atom's ${}^{3} \\mathrm{P}_{2}$ energy level is found to split into five sublevels, with the ratio of the intervals between adjacent sublevels being $9: 7: 5: 3$. Using the interval rule, determine the nuclear spin quantum number $I$ of this atom and the total atomic angular momentum quantum number $F$ corresponding to each sublevel. Find the nuclear spin quantum number $I$.",
 'solution': 'The rule for hyperfine level intervals is: for a given $J$ value, the interval between two adjacent hyperfine sublevels is proportional to the larger of the quantum numbers $F$ of the two sublevels.  \nSince the total atomic angular momentum quantum numbers are $F = I + J, I + J - 1, \\ldots, |I - J|$,  \nWhen $I \\geqslant J$, there are $2J+1$ sublevels; when $I < J$, there are $2I+1$ sublevels.  \nThere are 5 sublevels, and since $J = 2$, we have $2J+1 = 5$.  \

## Convert Physics QA → Text Format (Required for Curation)

In [16]:
def build_text(example):
    example["text"] = (
        "Problem:\n" + example["problem"].strip() +
        "\n\nSolution:\n" + example["solution"].strip()
    )
    return example

hf_dataset = hf_dataset.map(build_text)


## Save Dataset as JSONL

In [17]:
import os, json

os.makedirs("data/ugphysics_raw", exist_ok=True)
jsonl_path = "data/ugphysics_raw/ugphysics.jsonl"

with open(jsonl_path, "w", encoding="utf-8") as f:
    for row in hf_dataset:
        f.write(json.dumps({
            "text": row["text"],
            "domain": row["domain"],
            "subject": row["subject"],
            "topic": row["topic"],
        }, ensure_ascii=False) + "\n")

print("Saved dataset →", jsonl_path)
print("Total samples:", len(hf_dataset))


Saved dataset → data/ugphysics_raw/ugphysics.jsonl
Total samples: 915


## Load Dataset with NeMo Curator

In [18]:
from nemo_curator.datasets import DocumentDataset

raw_dataset = DocumentDataset.read_json(
    input_files="data/ugphysics_raw/ugphysics.jsonl"
)

raw_dataset.head()


Reading 1 files


,domain,subject,text,topic
0,Modern Physics,Atomic Physics,Problem:\nA certain atom's ${}^{3} \mathrm{P}_...,Atomic and Molecular Physics
1,Modern Physics,Atomic Physics,Problem:\nIf the energy of a photon is equal t...,Atomic and Molecular Physics
2,Modern Physics,Atomic Physics,"Problem:\nIn a hydrogen atom, when the transit...",Atomic and Molecular Physics
3,Modern Physics,Atomic Physics,Problem:\nThe Franck-Hertz experiment demonstr...,Atomic and Molecular Physics
4,Modern Physics,Atomic Physics,Problem:\nUsing the formula $\frac{|\Delta \la...,Atomic and Molecular Physics


## Text Cleaning & Normalization

In [19]:
from nemo_curator.modules.modify import Modify
from nemo_curator.modifiers import UnicodeReformatter, DocumentModifier
import re

class QuotationTagUnifier(DocumentModifier):
    def modify_document(self, text: str) -> str:
        text = text.replace("“", '"').replace("”", '"')
        text = text.replace("‘", "'").replace("’", "'")
        text = re.sub(r"<[^>]+>", "", text)
        return text

cleaned_dataset = Modify(QuotationTagUnifier())(raw_dataset)
cleaned_dataset = Modify(UnicodeReformatter())(cleaned_dataset)

cleaned_dataset = cleaned_dataset.persist()
cleaned_dataset.head()


,domain,subject,text,topic
0,Modern Physics,Atomic Physics,Problem:\nA certain atom's ${}^{3} \mathrm{P}_...,Atomic and Molecular Physics
1,Modern Physics,Atomic Physics,Problem:\nIf the energy of a photon is equal t...,Atomic and Molecular Physics
2,Modern Physics,Atomic Physics,"Problem:\nIn a hydrogen atom, when the transit...",Atomic and Molecular Physics
3,Modern Physics,Atomic Physics,Problem:\nThe Franck-Hertz experiment demonstr...,Atomic and Molecular Physics
4,Modern Physics,Atomic Physics,Problem:\nUsing the formula $\frac{|\Delta \la...,Atomic and Molecular Physics


## Quality Filtering

In [20]:
from nemo_curator.filters import WordCountFilter, RepeatingTopNGramsFilter
from nemo_curator import ScoreFilter

# Minimum word count
filtered_dataset = ScoreFilter(
    WordCountFilter(min_words=80),
    score_field="word_count"
)(cleaned_dataset)

# Repetition filter (3-grams)
filtered_dataset = ScoreFilter(
    RepeatingTopNGramsFilter(n=3, max_repeating_ngram_ratio=0.2)
)(filtered_dataset)

filtered_dataset = filtered_dataset.persist()
filtered_dataset.head()








/usr/local/lib/python3.10/dist-packages/dask_expr/_collection.py:4376: UserWarning: 
You did not provide metadata, so Dask is running your function on a small dataset to guess output types. It is possible that Dask will guess incorrectly.
To provide an explicit output types or to silence this message, please provide the `meta=` keyword, as described in the map or apply function that you are using.
  Before: .apply(func)
  After:  .apply(func, meta=('text', 'int64'))

  warnings.warn(meta_warning(meta))
/usr/local/lib/python3.10/dist-packages/dask_expr/_collection.py:4376: UserWarning: 
You did not provide metadata, so Dask is running your function on a small dataset to guess output types. It is possible that Dask will guess incorrectly.
To provide an explicit output types or to silence this message, please provide the `meta=` keyword, as described in the map or apply function that you are using.
  Before: .apply(func)
  After:  .apply(func, meta=('text', 'float64'))

  warnings.warn(me

,domain,subject,text,topic,word_count
0,Modern Physics,Atomic Physics,Problem:\nA certain atom's ${}^{3} \mathrm{P}_...,Atomic and Molecular Physics,180
2,Modern Physics,Atomic Physics,"Problem:\nIn a hydrogen atom, when the transit...",Atomic and Molecular Physics,96
3,Modern Physics,Atomic Physics,Problem:\nThe Franck-Hertz experiment demonstr...,Atomic and Molecular Physics,139
6,Modern Physics,Atomic Physics,"Problem:\nRegarding rare earth elements, which...",Atomic and Molecular Physics,105
8,Modern Physics,Atomic Physics,Problem:\nThe spectral term $\mathrm{D}_{5/2}$...,Atomic and Molecular Physics,129


## Language Detection

In [21]:
!wget https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin


--2026-01-12 15:37:29--  https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 13.249.182.62, 13.249.182.33, 13.249.182.39, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|13.249.182.62|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 131266198 (125M) [application/octet-stream]
Saving to: ‘lid.176.bin.4’

lid.176.bin.4       100%[===================>] 125.18M   338MB/s    in 0.4s    

2026-01-12 15:37:30 (338 MB/s) - ‘lid.176.bin.4’ saved [131266198/131266198]



In [22]:

from nemo_curator.filters import FastTextLangId
from nemo_curator import ScoreFilter

lang_filter = FastTextLangId("lid.176.bin")

language_dataset = ScoreFilter(
    lang_filter,
    score_field="language",
    score_type="object"
)(filtered_dataset)





## Add Unique Document IDs

In [23]:
from nemo_curator import AddId

add_id = AddId(
    id_field="id",
    id_prefix="EN_data",
    start_index=0
)

dataset_with_id = add_id(filtered_dataset).persist()
dataset_with_id.head()






,domain,subject,text,topic,word_count,language,id
0,Modern Physics,Atomic Physics,Problem:\nA certain atom's ${}^{3} \mathrm{P}_...,Atomic and Molecular Physics,180,"[1.0, N/A]",EN_data-0000000000
2,Modern Physics,Atomic Physics,"Problem:\nIn a hydrogen atom, when the transit...",Atomic and Molecular Physics,96,"[1.0, N/A]",EN_data-0000000001
3,Modern Physics,Atomic Physics,Problem:\nThe Franck-Hertz experiment demonstr...,Atomic and Molecular Physics,139,"[1.0, N/A]",EN_data-0000000002
6,Modern Physics,Atomic Physics,"Problem:\nRegarding rare earth elements, which...",Atomic and Molecular Physics,105,"[1.0, N/A]",EN_data-0000000003
8,Modern Physics,Atomic Physics,Problem:\nThe spectral term $\mathrm{D}_{5/2}$...,Atomic and Molecular Physics,129,"[1.0, N/A]",EN_data-0000000004


## Exact Deduplication

In [24]:
from nemo_curator import ExactDuplicates

exact_dedup = ExactDuplicates(
    id_field="id",
    text_field="text",
    hash_method="md5"
)

duplicates = exact_dedup(dataset_with_id)

dups_to_remove = duplicates.df.map_partitions(
    lambda df: df[df["_hashes"].duplicated(keep="first")]
)

print("Documents before:", len(dataset_with_id))
print("Duplicates detected:", len(dups_to_remove))

deduped_df = dataset_with_id.df[
    ~dataset_with_id.df["id"].isin(dups_to_remove["id"].compute())
]

print("Documents after:", len(deduped_df))


Documents before: 722
Duplicates detected: 0
Documents after: 722


## Save Final Curated Dataset

In [25]:
final_output_path = "./data/ugphysics_curated/final"
os.makedirs(final_output_path, exist_ok=True)

final_dataset = DocumentDataset(deduped_df)
final_dataset.to_json(final_output_path)

print("Final curated dataset saved to:", final_output_path)


Writing to disk complete for 1 partitions
Final curated dataset saved to: ./data/ugphysics_curated/final
